# 🔮 INTRACAPITAL — Google Colab Cloud Backend

Welcome to the cloud backend for **INTRACAPITAL** (Discover Businesses Hidden Inside Businesses).
Use this notebook to run the FastAPI services on Google Colab's cloud infrastructure (bypassing local disk and memory limits) and expose it to your local Streamlit client using **Localtunnel**.

### Step 1: Install Dependencies
Install the core backend frameworks, telemetry analytics libraries, local vector databases, and sentence transformers.

In [ ]:
!pip install fastapi uvicorn pydantic sentence-transformers pypdf pandas chromadb httpx python-dotenv

### Step 2: Clone Repository & Sync Files
Clone the INTRACAPITAL repository to load the modules (`backend/`, `tests/`, etc.).

In [ ]:
!git clone https://github.com/aadhyasharma270207-lang/intracapital.git
%cd intracapital

### Step 3: Configure Environment Variables
Create a `.env` file in the cloud environment. You can paste real IBM Granite credentials here, or leave them empty to run in Sandbox Simulation mode.

In [ ]:
with open('.env', 'w') as f:
    f.write('''
WATSONX_API_KEY=
WATSONX_PROJECT_ID=
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=ibm/granite-13b-instruct-v2
FASTAPI_INTERNAL_API_KEY=intracapital-secure-token-2026
''')
print("Cloud environment file created successfully!")

### Step 4: Expose FastAPI Port and Start Server
This cell runs two processes in the background:
1. **FastAPI application** on local port 8080.
2. **Localtunnel** to expose port 8080 to a public URL.

**Instructions:**
*   Run the cell below.
*   Locate the public link generated by Localtunnel (looks like `https://xxxx.loca.lt`).
*   Paste that link directly into the **FastAPI Endpoint URL** sidebar override inside your Streamlit application dashboard!

In [ ]:
# Install Localtunnel globally via npm
!npm install -g localtunnel

# Launch FastAPI server in the background
import subprocess
import time

print("Starting FastAPI server...")
fastapi_process = subprocess.Popen(["python", "-m", "uvicorn", "backend.api:app", "--host", "127.0.0.1", "--port", "8080"])
time.sleep(3)

print("Launching Tunnel link...")
tunnel_process = subprocess.Popen(["npx", "localtunnel", "--port", "8080"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Read tunnel URL
time.sleep(5)
url = ""
for line in iter(tunnel_process.stdout.readline, b''):
    line_str = line.decode('utf-8').strip()
    if "url is:" in line_str:
        url = line_str.split("url is:")[1].strip()
        print(f"\n🔮 SUCCESS: YOUR PUBLIC BACKEND ENDPOINT URL IS: {url}")
        print("Copy this link and paste it into the Streamlit sidebar field!\n")
        break

try:
    # Keep cell active to display logs
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping server processes...")
    fastapi_process.terminate()
    tunnel_process.terminate()